## Import libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from glob import glob
from obspy import read, UTCDateTime as utc
from obspy.geodetics.base import gps2dist_azimuth
from tqdm import tqdm

## Load catalog

In [2]:
event_catalog=pd.read_table("/home/msseo/works/72.2025_Imsil_swarm/event_catalog/Imsil_catalog.csv", sep=',')[8:9].reset_index(drop=True)
event_catalog

,Year,Month,Day,Hour,Minute,Second,Latitude,Longitude,Depth,Magnitude
0,2025,11,3,5,38,35,35.55,127.3,14,1.9


## Function to convert degrees to degrees/minutes/seconds

In [3]:
def degreetominute(angle):
    angle_deg=int(angle)
    angle_min=int(100*60*(angle-int(angle)))
    
    return [angle_deg, angle_min]

## Write input file

In [4]:
# Only change the names of following path and filename (do not need to touch rest of the code)
path_to_eventdir="/home/msseo/works/72.2025_Imsil_swarm/mainshock_waveforms/"
input_file="/home/msseo/works/72.2025_Imsil_swarm/1.HypoInv/PHS/Imsil.phs"

############### Keep codes below same regardless of the files that were used for picking ###############
f=open(input_file, 'w')
id_number=0
eventdir_list=sorted(glob(path_to_eventdir+"20*"))
for eventdir in tqdm(eventdir_list):
    eventid=eventdir.split("/")[-1]
    
    # Find event in the catalog that matches event directory
    for i in range(len(event_catalog)):
        year=str(event_catalog["Year"][i])
        month=str(event_catalog["Month"][i]).zfill(2)
        day=str(event_catalog["Day"][i]).zfill(2)
        hour=str(event_catalog["Hour"][i]).zfill(2)
        minute=str(event_catalog["Minute"][i]).zfill(2)
        second=str(int(event_catalog["Second"][i])).zfill(2)
        millisecond=str(int((event_catalog["Second"][i]-int(event_catalog["Second"][i]))*100)).zfill(2)
        
        origin_time_utc=utc(int(year), int(month), int(day), int(hour), int(minute), int(second))-9*3600
        year, month, day=origin_time_utc.year, origin_time_utc.month, origin_time_utc.day
        hour, minute, second=origin_time_utc.hour, origin_time_utc.minute, origin_time_utc.second        
    
        catalog_eventid=str(year)+str(month).zfill(2)+str(day).zfill(2)+str(hour).zfill(2)+str(minute).zfill(2)+str(second).zfill(2)
        # Write event origin time / latitude / longitude
        if eventid==catalog_eventid:
            lat_deg, lat_min=str(degreetominute(event_catalog["Latitude"][i])[0]), str(degreetominute(event_catalog["Latitude"][i])[1]).zfill(4)
            lon_deg, lon_min=str(degreetominute(event_catalog["Longitude"][i])[0]), str(degreetominute(event_catalog["Longitude"][i])[1]).zfill(4)
            eventinfo=catalog_eventid+millisecond+lat_deg+"N"+lat_min+lon_deg+"E"+lon_min
            
            f.write(eventinfo+"\n")
        
            # Get phase arrival information from each SAC file in the event directory
            sacfile_list=glob(eventdir+"/*.sac")
            for sacfile in sorted(sacfile_list)[::-1]:
                tr=read(sacfile)[0]
                station_name=sacfile.split("/")[-1].split(".")[2]
            
                # 1) Write P phase information

                try:
                    ptime=tr.stats.sac.a
                    evla, evlo=tr.stats.sac.evla, tr.stats.sac.evlo
                    stla, stlo=tr.stats.sac.stla, tr.stats.sac.stlo
                    epidist=gps2dist_azimuth(evla, evlo, stla, stlo)[0]/1000

                    if epidist<20:
                        p_weight="0"
                    elif 20<=epidist<50:
                        p_weight="1"
                    elif 50<=epidist<70:
                        p_weight="2"
                    elif 70<=epidist<100:
                        p_weight="3"
                    else:
                        p_weight="4"

                    origintime=tr.stats.starttime-tr.stats.sac.b+ptime
                    f.write(station_name.ljust(5))
                    f.write(tr.stats.network.ljust(4))
                    f.write(tr.stats.channel[:3].ljust(4))
                    f.write("IP".ljust(3))
                    f.write(p_weight)
                    f.write(str(origintime.year))
                    f.write(str(origintime.month).zfill(2))
                    f.write(str(origintime.day).zfill(2))
                    f.write(str(origintime.hour).zfill(2))
                    f.write(str(origintime.minute).zfill(2).ljust(3))
                    f.write(str(origintime.second).zfill(2))
                    f.write(str(origintime.microsecond).zfill(6)[:2])
                    f.write("\n")

                except:
                    pass

                # 2) Write S phase information

                try:
                    stime=tr.stats.sac.t0

                    evla, evlo=tr.stats.sac.evla, tr.stats.sac.evlo
                    stla, stlo=tr.stats.sac.stla, tr.stats.sac.stlo
                    epidist=gps2dist_azimuth(evla, evlo, stla, stlo)[0]/1000

                    if epidist<20:
                        s_weight="1"
                    elif 20<=epidist<50:
                        s_weight="2"
                    else:
                        s_weight="3"

                    origintime=tr.stats.starttime-tr.stats.sac.b+stime
                    f.write(station_name.ljust(5))
                    f.write(tr.stats.network.ljust(4))
                    f.write(tr.stats.channel[:3].ljust(4))
                    f.write("    ") # P wave phase name & weight missing
                    f.write(str(origintime.year))
                    f.write(str(origintime.month).zfill(2))
                    f.write(str(origintime.day).zfill(2))
                    f.write(str(origintime.hour).zfill(2))
                    f.write(str(origintime.minute).zfill(2).ljust(15))
                    f.write(str(origintime.second).zfill(2))
                    f.write(str(origintime.microsecond).zfill(6)[:2])
                    f.write("ES".ljust(3))
                    f.write(s_weight)
                    f.write("\n")

                except:
                    pass
    
    f.write(" "*66+"200"+f"{id_number}".zfill(3))
    f.write("\n")
    id_number += 1
            
f.close()

100%|██████████| 1/1 [00:00<00:00,  2.50it/s]
